In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"  # Windows 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

df = pd.read_csv("data/usdkrw_2021_2026.csv", parse_dates=["Date"], index_col="Date")
df = df.sort_index()

df.info()
df.head()


<class 'pandas.DataFrame'>
DatetimeIndex: 1494 entries, 2021-01-01 to 2026-09-30
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Adj Close  1494 non-null   float64
 1   Close      1494 non-null   float64
 2   High       1494 non-null   float64
 3   Low        1494 non-null   float64
 4   Open       1494 non-null   float64
 5   Volume     1494 non-null   int64  
dtypes: float64(5), int64(1)
memory usage: 81.7 KB


,Adj Close,Close,High,Low,Open,Volume
Date,,,,,,
2021-01-01,1084.650024,1084.650024,1088.500000,1083.500000,1083.500000,0
2021-01-04,1084.729980,1084.729980,1086.469971,1079.219971,1084.729980,0
2021-01-05,1086.619995,1086.619995,1089.729980,1081.670044,1085.609985,0
2021-01-06,1086.449951,1086.449951,1088.589966,1083.250000,1085.650024,0
2021-01-07,1086.949951,1086.949951,1096.000000,1084.829956,1085.250000,0


In [2]:
print("결측치:\n", df.isna().sum(), "\n")
print("중복 날짜:", df.index.duplicated().sum())
print("0 이하 값:", (df["Close"] <= 0).sum())

gaps = df.index.to_series().diff().dt.days
print("\n날짜 간격(일) 분포:\n", gaps.value_counts().sort_index())
print("\n간격이 큰 구간:\n", gaps[gaps >= 5])

결측치:
 Adj Close    0
Close        0
High         0
Low          0
Open         0
Volume       0
dtype: int64 

중복 날짜: 0
0 이하 값: 0

날짜 간격(일) 분포:
 Date
1.0    1190
2.0       3
3.0     299
5.0       1
Name: count, dtype: int64

간격이 큰 구간:
 Date
2025-04-22    5.0
Name: Date, dtype: float64


In [3]:
print(gaps[gaps >= 2])

Date
2021-01-04    3.0
2021-01-11    3.0
2021-01-18    3.0
2021-01-25    3.0
2021-02-01    3.0
             ... 
2026-08-31    3.0
2026-09-07    3.0
2026-09-14    3.0
2026-09-21    3.0
2026-09-28    3.0
Name: Date, Length: 303, dtype: float64


In [4]:
s = df[["Close"]].copy()   # 종가만 사용
s["Return"] = s["Close"].pct_change() * 100   # 일별 변화율(%)
s.describe()

,Close,Return
count,1494.000000,1493.000000
mean,1324.396072,0.016538
std,110.580981,0.608603
min,1084.650024,-3.438281
25%,1255.697479,-0.332676
50%,1335.544983,0.042000
75%,1401.629974,0.387047
max,1554.479980,2.120813


In [5]:
q1, q3 = s["Return"].quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 3 * iqr, q3 + 3 * iqr   # 기준: IQR의 3배
outliers = s[(s["Return"] < low) | (s["Return"] > high)]

print(f"기준 범위: {low:.2f}% ~ {high:.2f}%")
print("이상치 개수:", len(outliers))
print(outliers.sort_values("Return", key=abs, ascending=False).head(10))

기준 범위: -2.49% ~ 2.55%
이상치 개수: 4
                  Close    Return
Date                             
2022-12-01  1279.800049 -3.438281
2025-04-10  1443.949951 -2.838248
2022-11-14  1313.030029 -2.693106
2025-05-05  1398.900024 -2.522469


In [6]:
for d in outliers.sort_values("Return", key=abs, ascending=False).index:
    i = s.index.get_loc(d)
    print(s.iloc[i-1:i+2].round(2), "\n")

              Close  Return
Date                       
2022-11-30  1325.37   -1.10
2022-12-01  1279.80   -3.44
2022-12-02  1303.25    1.83 

              Close  Return
Date                       
2025-04-09  1486.13    1.10
2025-04-10  1443.95   -2.84
2025-04-11  1452.04    0.56 

              Close  Return
Date                       
2022-11-11  1349.37   -1.55
2022-11-14  1313.03   -2.69
2022-11-15  1324.08    0.84 

              Close  Return
Date                       
2025-05-02  1435.10    0.80
2025-05-05  1398.90   -2.52
2025-05-06  1374.37   -1.75 



In [7]:
print("최저:", s["Close"].idxmin().date(), round(s["Close"].min(), 2))
print("최고:", s["Close"].idxmax().date(), round(s["Close"].max(), 2))

최저: 2021-01-01 1084.65
최고: 2026-06-08 1554.48


In [8]:
s.to_csv("data/usdkrw_clean.csv")